# JAX & Flax

**Domain:** AI/ML Tooling  ·  **recommended addition**  ·  **runnable:** yes

A refresher on Google's array/autodiff stack: **JAX** (NumPy + autodiff + JIT + auto-vectorization, all composable) and **Flax** (the neural-network library built on top of it).

## 1. What & Why

**JAX** is NumPy with three superpowers bolted on as *composable function transformations*:

- `grad` — exact autodiff of any pure Python/NumPy function.
- `jit` — trace a function to an XLA graph and compile it (CPU/GPU/TPU) for large speedups.
- `vmap` / `pmap` — auto-vectorize or shard a function across a batch / multiple devices without rewriting it.

Because these are just functions that take a function and return a function, you can stack them: `jit(grad(vmap(f)))` is ordinary code. That composability is the whole pitch.

**Flax** is the neural-network layer on top. JAX gives you arrays and gradients but no notion of "a Linear layer with weights"; Flax (its current `flax.nnx` API) gives you `Module`s that hold parameters as ordinary Python attributes, plus an `Optimizer` wrapper around [Optax](https://optax.readthedocs.io/).

**Reach for it when** you want maximum performance on TPUs/GPUs, research code where you need to differentiate or vectorize through *anything* (physics sims, custom losses, meta-learning, scientific computing), or you value the functional/explicit style. **Skip it when** you want the largest ecosystem of pretrained models and tutorials, or a gentle on-ramp — that's still PyTorch territory.

## 2. Mental Model

**JAX = NumPy that you compile, and a set of verbs you wrap around your math.**

You write a plain mathematical function `f(params, x) -> scalar`. Then you don't *edit* `f` to make it differentiable or fast or batched — you wrap it:

```
f                      # the math, written once, pure (no side effects)
grad(f)                # → function returning ∂f/∂params
jit(grad(f))           # → same, but compiled by XLA
vmap(jit(grad(f)))     # → same, but auto-batched over the leading axis
```

Two consequences that drive everything else:

1. **Functions must be pure.** `jit` *traces* your function once with abstract placeholder values to build a graph, then reuses that graph. Side effects (printing, mutating globals, Python `if` on array values) happen only during the trace, not on later calls — which is the source of most JAX surprises.
2. **No in-place mutation.** Arrays are immutable. There is no `model.train()` flag or `param -= grad`; you take old state in and return new state out. Flax's `nnx` API hides some of this behind objects, but underneath, training is `new_params = update(old_params, grads)`.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **`jax.numpy` (`jnp`)** | Drop-in NumPy API that produces traceable, device-resident arrays. |
| **Tracing** | `jit`/`grad` run your function on abstract values to record the operations, building an XLA graph that's cached and reused. |
| **Pure functions** | No side effects, output depends only on inputs. Required for the transforms to be correct. |
| **PyTree** | Any nested structure of lists/tuples/dicts whose leaves are arrays. Params, optimizer state, and batches are all pytrees; `jax.tree.map` walks them. |
| **Explicit PRNG** | No global random state. You hold a `key`, and `jax.random.split` it to get fresh keys — randomness is reproducible and parallel-safe. |
| **`grad` / `value_and_grad`** | Reverse-mode autodiff of a scalar-output function w.r.t. its first argument (by default). |
| **`jit`** | Compile-and-cache. First call traces + compiles (slow); later calls with the same input *shapes/dtypes* hit the cache. |
| **`vmap`** | Add a batch dimension to a function written for a single example. `in_axes` says which arg is batched. |
| **Flax `nnx.Module`** | A layer/model whose parameters live as attributes; composes like normal Python objects. |
| **Optax** | Standalone gradient-transformation library (Adam, SGD, schedules, clipping) that JAX/Flax use as the optimizer. |

## 4. Setup

CPU is enough for this notebook. The base `jax` wheel is CPU-only; for accelerators you install a matching build (`jax[cuda12]` for NVIDIA, or use a TPU runtime).

```bash
pip install -U "jax" flax optax        # CPU
pip install -U "jax[cuda12]" flax optax # NVIDIA GPU (CUDA 12)
```

In [1]:
# If JAX/Flax aren't installed in this kernel, uncomment:
# %pip install -q -U jax flax optax
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")  # keep this refresher CPU-only & deterministic

import jax, jax.numpy as jnp
import flax, optax
from jax import grad, jit, vmap

print(f"jax   {jax.__version__}")
print(f"flax  {flax.__version__}")
print(f"optax {optax.__version__}")
print("devices:", jax.devices())

jax   0.10.2
flax  0.12.7
optax 0.2.8
devices: [CpuDevice(id=0)]


## 5. Worked Examples

Three tiny, CPU-friendly examples: (1) the `grad`/`jit`/`vmap` core on a linear regression, (2) what `jit` actually buys you, and (3) a real Flax `nnx` model trained with Optax.

### 5.1 The core: `grad` + `jit` + `vmap`

Fit `y = Xw + b` by gradient descent. Note the shape: we write the loss as plain math, then `jit(grad(...))` gives us a fast, compiled gradient function. Parameters are just a tuple — a pytree.

In [2]:
# Synthetic linear data
X = jax.random.normal(jax.random.PRNGKey(0), (200, 3))
true_w, true_b = jnp.array([2.0, -1.0, 0.5]), 0.3
noise = 0.1 * jax.random.normal(jax.random.PRNGKey(1), (200,))
y = X @ true_w + true_b + noise

def loss(params, X, y):
    w, b = params
    pred = X @ w + b
    return jnp.mean((pred - y) ** 2)        # plain math, pure function

grad_loss = jit(grad(loss))                 # compiled gradient w.r.t. params (arg 0)
params = (jnp.zeros(3), 0.0)                # a pytree: (w, b)
lr = 0.1
for step in range(201):
    g = grad_loss(params, X, y)
    params = jax.tree.map(lambda p, gi: p - lr * gi, params, g)  # immutable update
    if step % 50 == 0:
        print(f"step {step:3d}  loss {loss(params, X, y):.4f}")

w, b = params
print("\nlearned w:", jnp.round(w, 2), " b:", round(float(b), 2))
print("true    w:", true_w, " b:", true_b)

# vmap: a single-example predictor, auto-batched over rows of X
predict_one = lambda x, w, b: jnp.dot(x, w) + b
batched = vmap(predict_one, in_axes=(0, None, None))   # batch arg 0, broadcast w & b
print("vmap preds[:3]:", jnp.round(batched(X[:3], w, b), 2))

step   0  loss 3.1872
step  50  loss 0.0115
step 100  loss 0.0115
step 150  loss 0.0115
step 200  loss 0.0115

learned w: [ 2.01 -1.    0.5 ]  b: 0.31
true    w: [ 2.  -1.   0.5]  b: 0.3
vmap preds[:3]: [ 1.3299999 -0.51      -0.84     ]


### 5.2 What `jit` buys you

`jit` traces the function once into an XLA graph that fuses the elementwise ops, then reuses the compiled graph. We warm up first (compilation is a one-time cost you don't want to time) and use `block_until_ready()` because JAX dispatch is asynchronous. Exact numbers vary by machine, but the compiled version is consistently faster.

In [3]:
import time

def f(x):
    for _ in range(50):           # a chain of ops XLA can fuse into one kernel
        x = jnp.tanh(x) + x * 0.5
    return x.sum()

x = jax.random.normal(jax.random.PRNGKey(0), (4000,))
f_jit = jit(f)

f(x).block_until_ready()          # warm up eager
f_jit(x).block_until_ready()      # warm up: triggers tracing + compilation

def bench(fn, n=50):
    t0 = time.perf_counter()
    for _ in range(n):
        fn(x).block_until_ready()
    return (time.perf_counter() - t0) / n * 1e3   # ms per call

print(f"eager : {bench(f):6.3f} ms/call")
print(f"jit   : {bench(f_jit):6.3f} ms/call")

eager :  0.592 ms/call
jit   :  0.138 ms/call


### 5.3 A Flax model with `nnx` + Optax

Flax's current `nnx` API lets you write models as ordinary Python objects: parameters live as attributes, and you train with an `nnx.Optimizer` wrapping an Optax optimizer. Here: a 2-layer MLP learning to classify points inside vs. outside a circle.

In [4]:
from flax import nnx

class MLP(nnx.Module):
    def __init__(self, din, dhidden, dout, *, rngs):
        self.l1 = nnx.Linear(din, dhidden, rngs=rngs)
        self.l2 = nnx.Linear(dhidden, dout, rngs=rngs)
    def __call__(self, x):
        return self.l2(nnx.relu(self.l1(x)))

# Toy task: is the point outside a circle of radius^2 = 1.5?
X = jax.random.normal(jax.random.PRNGKey(0), (256, 2))
y = (jnp.sum(X ** 2, axis=1) > 1.5).astype(jnp.int32)

model = MLP(2, 16, 2, rngs=nnx.Rngs(0))
optimizer = nnx.Optimizer(model, optax.adam(1e-2), wrt=nnx.Param)

def loss_fn(model, X, y):
    logits = model(X)
    return optax.softmax_cross_entropy_with_integer_labels(logits, y).mean()

@nnx.jit                                  # nnx-aware jit: handles the model object
def train_step(model, optimizer, X, y):
    loss, grads = nnx.value_and_grad(loss_fn)(model, X, y)
    optimizer.update(model, grads)        # in-place on the nnx object
    return loss

for epoch in range(201):
    loss = train_step(model, optimizer, X, y)
    if epoch % 50 == 0:
        acc = jnp.mean(jnp.argmax(model(X), axis=1) == y)
        print(f"epoch {epoch:3d}  loss {loss:.4f}  acc {acc:.3f}")

epoch   0  loss 0.8063  acc 0.551
epoch  50  loss 0.3519  acc 0.945
epoch 100  loss 0.1480  acc 0.984
epoch 150  loss 0.0879  acc 0.984
epoch 200  loss 0.0617  acc 0.996


## 6. Gotchas & Pitfalls

- **Side effects only fire during tracing.** A `print(x)` inside a `jit`-ed function prints once (at trace time), not every call. Use `jax.debug.print("{}", x)` to print actual runtime values.
- **No Python control flow on traced values.** `if x > 0:` on an array inside `jit` fails — the tracer has no concrete value. Use `jnp.where`, or `jax.lax.cond` / `jax.lax.scan` for data-dependent branches and loops.
- **No in-place assignment.** `x[0] = 1` raises. Use the functional `x = x.at[0].set(1)`.
- **Recompilation on shape changes.** `jit` caches per input *shape/dtype*. Variable-length batches re-trace every call and kill performance — pad to fixed shapes. Pass true constants via `static_argnums` (but each distinct value recompiles).
- **PRNG keys must be split, not reused.** Reusing a key gives identical "random" draws. Always `key, sub = jax.random.split(key)`. There is no global seed.
- **float32 by default.** JAX silently uses 32-bit floats even from float64 inputs. For double precision set `jax.config.update("jax_enable_x64", True)` *before* creating arrays.
- **Async dispatch fools benchmarks.** Operations return immediately; the work runs later. Call `.block_until_ready()` before measuring time, and warm up once to exclude compilation.
- **Flax API churn.** `flax.linen` (the `@nn.compact`, explicit `params` dict style) and the newer `flax.nnx` (object/attribute style, used here) coexist. Match tutorials to your version; new code should prefer `nnx`.

## 7. When to Use vs Alternatives

| Option | Strengths | Weaknesses | Pick it when |
|---|---|---|---|
| **JAX + Flax** | Best-in-class TPU support, composable transforms (`grad`/`jit`/`vmap`/`pmap`), functional purity, great for research & scientific computing | Smaller ecosystem, steeper learning curve, functional style is a mental shift, fewer pretrained models | TPU training, custom math/physics, meta-learning, you want explicit control |
| **PyTorch** | Huge ecosystem, most tutorials & pretrained weights, eager-by-default debugging, `torch.compile` closes much of the speed gap | Less elegant multi-device story than `pmap`, autodiff less composable | Default for most applied DL, production, and learning |
| **TensorFlow / Keras** | Mature serving/mobile (TFLite, TF Serving), Keras high-level API | Lower research momentum, heavier API | Existing TF stack, edge/mobile deployment |
| **Plain NumPy** | Simplest, zero deps | No autodiff, no GPU, no JIT | Small CPU array work with no gradients |

Rule of thumb: **PyTorch is the safe default; reach for JAX when you need TPUs, large-scale parallelism, or to differentiate/vectorize through non-standard code.** Note JAX and Flax also underpin much of Google's own model code and libraries like Hugging Face's JAX/Flax model ports.

## 8. Resources

- **JAX docs** — https://docs.jax.dev/ (start with "JAX in 100 seconds" and "Thinking in JAX").
- **The Sharp Bits** — https://docs.jax.dev/en/latest/notebooks/Common_Gotchas_in_JAX.html — the canonical list of purity/PRNG/control-flow traps; read it once and the gotchas above will stick.
- **Flax (`nnx`) docs** — https://flax.readthedocs.io/ — the current neural-net API used in example 5.3.
- **Optax docs** — https://optax.readthedocs.io/ — optimizers and gradient transforms used by JAX/Flax.
- **Autodidax** — https://docs.jax.dev/en/latest/autodidax.html — JAX's transforms implemented from scratch, if you want to see how tracing actually works.